In [1]:
"""
Recompute test-set R2 for all 5 district weather-LSTM models using two methods:

  1) "blended" R2  - R2 computed separately for each of the 11 target features,
     then averaged (this is the exact method/number train_district.py reports
     and saves into each district's config.json / test_metrics_by_feature.csv)

  2) "overall" R2   - a single R2 computed on the ENTIRE flattened y_true/y_pred
     arrays, pooling all 11 features x all test windows x all 3 forecast days
     together into one r2_score() call

RMSE is reported alongside R2 in the same two ways (per-feature, in each
feature's original units, and overall on the flattened array).

Requires: torch, scikit-learn, joblib, pandas, numpy
    pip install torch scikit-learn joblib pandas numpy
"""
import os
import json

import numpy as np
import pandas as pd
import joblib
import torch
import torch.nn as nn
from sklearn.metrics import r2_score, mean_squared_error

# ----------------------------------------------------------------------
# PATHS
# ----------------------------------------------------------------------
BASE_DIR = r'C:\Academics\project\final year project'
CSV_DIR = r'C:\Academics\project\final year project\Data\Weather Data\V2'
MODEL_DIR = r'C:\Academics\project\final year project\models'

# district -> (weather csv filename, saved model filename)
DISTRICTS = {
    "Latur":      ("latur_weather_2000_2025.csv",      "latur_weather_lstm_pytorch.pt"),
    "Beed":       ("beed_weather_2000_2025.csv",       "beed_weather_lstm_pytorch.pt"),
    "Osmanabad":  ("osmanabad_weather_2000_2025.csv",  "osmanabad_weather_lstm_pytorch.pt"),
    "Solapur":    ("solapur_weather_2000_2025.csv",    "solapur_weather_lstm_pytorch.pt"),
    "Ahmednagar": ("ahmednagar_weather_2000_2025.csv", "ahmednagar_weather_lstm_pytorch.pt"),
}


class DirectLSTM(nn.Module):
    def __init__(self, n_input, n_target, horizon, lstm_units, num_layers, dense_units, dropout):
        super().__init__()
        self.lstm = nn.LSTM(n_input, lstm_units, num_layers=num_layers, batch_first=True,
                             dropout=dropout if num_layers > 1 else 0.0)
        self.horizon = horizon
        self.n_target = n_target
        self.drop_lstm = nn.Dropout(dropout)
        self.fc1 = nn.Linear(lstm_units + horizon * 2, dense_units)
        self.drop1 = nn.Dropout(dropout)
        self.fc2 = nn.Linear(dense_units, dense_units // 2)
        self.drop2 = nn.Dropout(dropout)
        self.fc3 = nn.Linear(dense_units // 2, horizon * n_target)
        self.act = nn.ReLU()

    def forward(self, seq, cal):
        _, (h, _) = self.lstm(seq)
        enc = self.drop_lstm(h[-1])
        cal_flat = cal.reshape(cal.size(0), -1)
        x = torch.cat([enc, cal_flat], dim=1)
        x = self.drop1(self.act(self.fc1(x)))
        x = self.drop2(self.act(self.fc2(x)))
        x = self.fc3(x)
        return x.view(-1, self.horizon, self.n_target)


def calendar_feats(index):
    doy = np.asarray(index.dayofyear)
    leap = np.asarray(index.is_leap_year)
    frac = doy / np.where(leap, 366, 365)
    return np.stack([np.sin(2 * np.pi * frac), np.cos(2 * np.pi * frac)], axis=1)


def make_windows(in_arr, out_arr, cal_arr, lookback, horizon):
    Xs, Ys, Cs = [], [], []
    n = len(in_arr)
    for i in range(n - lookback - horizon + 1):
        Xs.append(in_arr[i:i + lookback])
        Ys.append(out_arr[i + lookback:i + lookback + horizon])
        Cs.append(cal_arr[i + lookback:i + lookback + horizon])
    return (np.array(Xs, dtype=np.float32), np.array(Ys, dtype=np.float32), np.array(Cs, dtype=np.float32))


def make_windows_with_context(in_ctx, out_ctx, cal_ctx, in_arr, out_arr, cal_arr, lookback, horizon):
    full_in = np.concatenate([in_ctx[-lookback:], in_arr], axis=0)
    full_out = np.concatenate([out_ctx[-lookback:], out_arr], axis=0)
    full_cal = np.concatenate([cal_ctx[-lookback:], cal_arr], axis=0)
    return make_windows(full_in, full_out, full_cal, lookback, horizon)


def inverse_target(scaled_seq, scaler, log_cols, feature_names):
    n, h, f = scaled_seq.shape
    flat = scaled_seq.reshape(-1, f)
    inv = scaler.inverse_transform(flat).reshape(n, h, f)
    for c in log_cols:
        j = feature_names.index(c)
        inv[:, :, j] = np.expm1(inv[:, :, j])
    return inv


def preprocess_and_predict(district, csv_name, model_name):
    csv_path = os.path.join(CSV_DIR, csv_name)
    mdir = os.path.join(MODEL_DIR, district)
    model_path = os.path.join(mdir, model_name)

    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"Weather CSV not found: {csv_path}")
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"Model file not found: {model_path}")

    config = json.load(open(os.path.join(mdir, "config.json")))

    TRAIN_END, VAL_END = config["train_end"], config["val_end"]
    LOOKBACK, HORIZON = config["lookback"], config["horizon"]
    LOG_COLS = config["log_transformed_columns"]
    TARGET_FEATURES = config["target_features"]
    INPUT_FEATURES = config["input_features"]

    raw = pd.read_csv(csv_path, parse_dates=["DATE"]).sort_values("DATE").reset_index(drop=True)
    df = raw.copy()
    all_nan_mask = df[TARGET_FEATURES].isna().all(axis=1)
    if all_nan_mask.any():
        last_valid_idx = df.index[~all_nan_mask].max()
        df = df.loc[:last_valid_idx].reset_index(drop=True)
    df = df.set_index("DATE")

    clim_col = "Cloud_Amount_percent"
    train_slice = df.loc[:TRAIN_END, clim_col]
    clim = train_slice.groupby(train_slice.index.dayofyear).mean()
    clim_ext = pd.concat([clim, clim, clim])
    clim_smooth = clim_ext.rolling(7, center=True, min_periods=1).mean().iloc[len(clim):2 * len(clim)]
    clim_smooth.index = clim.index
    doy_all = df.index.dayofyear
    fill_values = doy_all.map(clim_smooth)
    mask = df[clim_col].isna()
    df.loc[mask, clim_col] = fill_values[mask.values]
    other_cols = [c for c in TARGET_FEATURES if c != clim_col]
    df[other_cols] = df[other_cols].interpolate(method="time", limit_direction="both")
    assert df.isna().sum().sum() == 0

    df_model = df.copy()
    for c in LOG_COLS:
        df_model[c] = np.log1p(df_model[c])

    df_model["Rain_roll7"] = df["Rainfall_mm"].rolling(7, min_periods=1).sum()
    df_model["Rain_roll14"] = df["Rainfall_mm"].rolling(14, min_periods=1).sum()
    df_model["Rain_roll30"] = df["Rainfall_mm"].rolling(30, min_periods=1).sum()
    df_model["Humid_roll7"] = df["Relative_Humidity_percent"].rolling(7, min_periods=1).mean()
    df_model["Solar_roll7"] = df["Solar_Radiation"].rolling(7, min_periods=1).mean()
    df_model["Wind_roll7"] = df["Wind_Speed_mps"].rolling(7, min_periods=1).mean()
    df_model["Wind50_roll7"] = df["Wind_Speed_50m_mps"].rolling(7, min_periods=1).mean()
    df_model["Cloud_roll7"] = df["Cloud_Amount_percent"].rolling(7, min_periods=1).mean()

    train_df = df_model.loc[:TRAIN_END]
    val_df = df_model.loc[pd.Timestamp(TRAIN_END) + pd.Timedelta(days=1):VAL_END]
    test_df = df_model.loc[pd.Timestamp(VAL_END) + pd.Timedelta(days=1):]

    in_scaler = joblib.load(os.path.join(mdir, "in_scaler.joblib"))
    out_scaler = joblib.load(os.path.join(mdir, "out_scaler.joblib"))
    # NOTE: use .transform() only here, never .fit_transform() - the scalers
    # were already fit on the training split when the model was trained, and
    # refitting them here would silently change the scaling and invalidate
    # any comparison with the originally reported metrics.
    train_in = in_scaler.transform(train_df[INPUT_FEATURES])
    val_in = in_scaler.transform(val_df[INPUT_FEATURES])
    test_in = in_scaler.transform(test_df[INPUT_FEATURES])
    train_out = out_scaler.transform(train_df[TARGET_FEATURES])
    val_out = out_scaler.transform(val_df[TARGET_FEATURES])
    test_out = out_scaler.transform(test_df[TARGET_FEATURES])

    cal_train = calendar_feats(train_df.index)
    cal_val = calendar_feats(val_df.index)
    cal_test = calendar_feats(test_df.index)

    trainval_in = np.concatenate([train_in, val_in], axis=0)
    trainval_out = np.concatenate([train_out, val_out], axis=0)
    trainval_cal = np.concatenate([cal_train, cal_val], axis=0)
    Xtest, Ytest, Ctest = make_windows_with_context(trainval_in, trainval_out, trainval_cal,
                                                     test_in, test_out, cal_test, LOOKBACK, HORIZON)

    ckpt = torch.load(model_path, map_location="cpu", weights_only=False)
    mc = ckpt["model_config"]
    model = DirectLSTM(mc["n_input"], mc["n_target"], mc["horizon"], mc["lstm_units"],
                        mc["num_layers"], mc["dense_units"], mc["dropout"])
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()

    with torch.no_grad():
        pred_scaled = model(torch.from_numpy(Xtest), torch.from_numpy(Ctest)).numpy()

    y_true = inverse_target(Ytest, out_scaler, LOG_COLS, TARGET_FEATURES)
    y_pred = inverse_target(pred_scaled, out_scaler, LOG_COLS, TARGET_FEATURES)
    return y_true, y_pred, TARGET_FEATURES


def main():
    results = []
    for district, (csv_name, model_name) in DISTRICTS.items():
        print(f"Processing {district} ...")
        y_true, y_pred, TARGET_FEATURES = preprocess_and_predict(district, csv_name, model_name)

        # ---- Method 1: per-feature R2, then averaged ("blended R2") ----
        per_feat_r2 = {}
        per_feat_rmse = {}
        for j, feat in enumerate(TARGET_FEATURES):
            a, p = y_true[:, :, j].reshape(-1), y_pred[:, :, j].reshape(-1)
            per_feat_r2[feat] = r2_score(a, p)
            per_feat_rmse[feat] = float(np.sqrt(mean_squared_error(a, p)))
        blended_r2 = float(np.mean(list(per_feat_r2.values())))
        # NOTE: features have different units (mm, %, degC, m/s ...), so the
        # averaged RMSE is only a rough summary - the per-feature RMSEs are
        # the meaningful numbers.
        blended_rmse = float(np.mean(list(per_feat_rmse.values())))

        # ---- Method 2: single R2 on the ENTIRE flattened array
        #      (all features + all test windows + all horizon steps pooled) ----
        a_all = y_true.reshape(-1)
        p_all = y_pred.reshape(-1)
        overall_r2 = float(r2_score(a_all, p_all))
        overall_rmse = float(np.sqrt(mean_squared_error(a_all, p_all)))

        results.append({
            "district": district,
            "blended_r2_avg_of_features": round(blended_r2, 4),
            "overall_r2_entire_array": round(overall_r2, 4),
            "blended_rmse_avg_of_features": round(blended_rmse, 4),
            "overall_rmse_entire_array": round(overall_rmse, 4),
            **{f"r2_{feat}": round(v, 4) for feat, v in per_feat_r2.items()},
            **{f"rmse_{feat}": round(v, 4) for feat, v in per_feat_rmse.items()},
        })
        print(f"  blended (avg of 11 features) R2 = {blended_r2:.4f}")
        print(f"  overall (entire flattened array) R2 = {overall_r2:.4f}")
        print(f"  blended (avg of 11 features) RMSE = {blended_rmse:.4f}")
        print(f"  overall (entire flattened array) RMSE = {overall_rmse:.4f}")
        print("  per-feature RMSE (original units):")
        for feat, v in per_feat_rmse.items():
            print(f"    {feat:<30s} {v:.4f}")
        print()

    df = pd.DataFrame(results)
    print(df[["district", "blended_r2_avg_of_features", "overall_r2_entire_array",
              "blended_rmse_avg_of_features", "overall_rmse_entire_array"]].to_string(index=False))

    out_csv = os.path.join(BASE_DIR, "r2_all_districts.csv")
    df.to_csv(out_csv, index=False)
    print(f"\nSaved full results (including per-feature R2 and RMSE) to: {out_csv}")


if __name__ == "__main__":
    main()

Processing Latur ...


c:\Users\vyshn\miniconda3\envs\main\lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.8.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


  blended (avg of 11 features) R2 = 0.7377
  overall (entire flattened array) R2 = 0.9537
  blended (avg of 11 features) RMSE = 3.8272
  overall (entire flattened array) RMSE = 6.4328
  per-feature RMSE (original units):
    Rainfall_mm                    4.7211
    Temperature_C                  1.1495
    Temperature_Max_C              1.5266
    Temperature_Min_C              1.3779
    Relative_Humidity_percent      6.1231
    Wind_Speed_mps                 0.9387
    Surface_Pressure_kPa           0.1645
    Solar_Radiation                3.1378
    Dew_Point_C                    2.4083
    Cloud_Amount_percent           19.2792
    Wind_Speed_50m_mps             1.2724

Processing Beed ...
  blended (avg of 11 features) R2 = 0.7504
  overall (entire flattened array) R2 = 0.9557
  blended (avg of 11 features) RMSE = 3.7532
  overall (entire flattened array) RMSE = 6.2725
  per-feature RMSE (original units):
    Rainfall_mm                    4.8972
    Temperature_C               

c:\Users\vyshn\miniconda3\envs\main\lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.8.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\vyshn\miniconda3\envs\main\lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.8.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


  blended (avg of 11 features) R2 = 0.7246
  overall (entire flattened array) R2 = 0.9530
  blended (avg of 11 features) RMSE = 3.8766
  overall (entire flattened array) RMSE = 6.4792
  per-feature RMSE (original units):
    Rainfall_mm                    5.0196
    Temperature_C                  1.1621
    Temperature_Max_C              1.5679
    Temperature_Min_C              1.3698
    Relative_Humidity_percent      6.1422
    Wind_Speed_mps                 0.9385
    Surface_Pressure_kPa           0.1558
    Solar_Radiation                3.1335
    Dew_Point_C                    2.5390
    Cloud_Amount_percent           19.3496
    Wind_Speed_50m_mps             1.2646

Processing Solapur ...
  blended (avg of 11 features) R2 = 0.7358
  overall (entire flattened array) R2 = 0.9545
  blended (avg of 11 features) RMSE = 3.7281
  overall (entire flattened array) RMSE = 6.3431
  per-feature RMSE (original units):
    Rainfall_mm                    4.4376
    Temperature_C            

c:\Users\vyshn\miniconda3\envs\main\lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.8.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\vyshn\miniconda3\envs\main\lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.8.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


  blended (avg of 11 features) R2 = 0.7485
  overall (entire flattened array) R2 = 0.9533
  blended (avg of 11 features) RMSE = 3.9198
  overall (entire flattened array) RMSE = 6.4182
  per-feature RMSE (original units):
    Rainfall_mm                    7.2985
    Temperature_C                  1.1213
    Temperature_Max_C              1.4703
    Temperature_Min_C              1.4003
    Relative_Humidity_percent      5.6671
    Wind_Speed_mps                 0.9568
    Surface_Pressure_kPa           0.1483
    Solar_Radiation                2.9700
    Dew_Point_C                    2.2309
    Cloud_Amount_percent           18.6027
    Wind_Speed_50m_mps             1.2523

  district  blended_r2_avg_of_features  overall_r2_entire_array  blended_rmse_avg_of_features  overall_rmse_entire_array
     Latur                      0.7377                   0.9537                        3.8272                     6.4328
      Beed                      0.7504                   0.9557          